Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Homework 3b: similarity search with light-curve embeddings

**Task:** a non-DRH colleague gives you the coordinates of an object and asks you to find DP2 objects similar to it.

**Plan:**
1. Find the DP2 object at these coordinates
2. Embed DP2 light curves in a small sky region
3. Find the nearest neighbours of the given object
4. Share back their object IDs, cosine distances, and embeddings

In this notebook, the object is [AT2025tsz](https://www.wis-tns.org/object/2025tsz), a transient in the DP2 ELAIS-S1 deep drilling field.

We've selected AT2025tsz, a SN Ia candidate, with the [TNS EDP2 explorer](https://trivialtz.github.io/tns-edp2-explorer/).

If you'd like to solve it yourself, `Notebook_3a_LightCurveFeatures` may help.

> [!NOTE]
> This notebook uses Rubin DP2 data, so it can be run by Rubin data rights holders (DRHs) only, e.g. on the [Rubin Science Platform](https://data.lsst.cloud) with the latest weekly image.
> Since we would like to share the notebook and its outputs with everyone, we never print or plot individual rows of the DP2 catalog: all the outputs are either catalog metadata or aggregated statistics.

## Solution

In [ ]:
# lsdb is pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb
# light-curve and its embedding dependencies are not pre-installed on RSP
%pip install -qq light-curve onnxruntime huggingface-hub

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import dask
import light_curve as licu
import lsdb
import nested_pandas as npd
import numpy as np
import pandas as pd
import pyarrow as pa
from dask.distributed import Client
from light_curve.embed import ATCAT
from light_curve.light_curve_py.warnings import ExperimentalWarning
from scipy.spatial.distance import cdist

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)
dask.config.set({"dataframe.convert-string": False})
# Hugging Face Hub warns about model downloads without a token, we silence it.
# Environment variables are inherited by Dask workers, so we set them before starting the client.
os.environ["HF_HUB_VERBOSITY"] = "error"
# Uncomment to use your Hugging Face token for faster model downloads
# os.environ["HF_TOKEN"] = "hf_..."


def silence_warnings():
    """Silence warnings of light_curve.embed, which is experimental and may change in future versions."""
    warnings.filterwarnings("ignore", category=ExperimentalWarning)


silence_warnings()

### Setup Dask cluster

As in `Notebook_3a_LightCurveFeatures`, we use four workers with one thread each, and let each worker use four threads for `light-curve`.

In [ ]:
client = Client(
    n_workers=4,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)
# Dask workers don't inherit warning filters, so we set them there too
client.run(silence_warnings)
print(f"Dask is running at: {client.dashboard_link}")

### Query object

This is all we get from our colleague: the name and the coordinates from TNS.

In [ ]:
QUERY_NAME = "AT2025tsz"
QUERY_RA, QUERY_DEC = 9.76838, -42.96739

### Open the DP2 DiaObject catalog and filter the light curves

For demonstration purposes, we filter aggressively and select bright objects with a lot of detections. We get the number of detections from `nDiaSources`: unlike the forced photometry, which is measured on every visit, DiaSources are the observations which passed the detection threshold on the difference images. We start with this cut, `nDiaSources >= 20`, as it is the cheapest one: `nDiaSources` is a plain column, so we pass the cut as `filters` to `open_catalog()`, and the rows are skipped already when the Parquet files are read.

For the light curves, as in `Notebook_3a_LightCurveFeatures`, we use forced photometry on the difference images with corrected errors, and filter out flagged observations. We keep objects with at least 20 observations, and sort observations by time, as required by `light-curve`.

In [ ]:
# DP2 DiaObject collection on RSP, check https://lsdb.io/dp2 for the paths on other platforms
dia_object = lsdb.open_catalog(
    "/rubin/lsdb_data/dp2/dia_object_collection",
    # coordinates and _healpix_29 are always loaded
    columns=[
        "diaObjectId",
        "nDiaSources",
        "diaObjectForcedSource.band",
        "diaObjectForcedSource.midpointMjdTai",
        "diaObjectForcedSource.psfDiffFlux",
        "diaObjectForcedSource.psfDiffFluxErr_corrected",
        "diaObjectForcedSource.psfDiffFlux_flag",
        "diaObjectForcedSource.psfDiffFluxErr_corrected_flag",
        "diaObjectForcedSource.diff_PixelFlags_nodataCenter",
        "diaObjectForcedSource.pixelFlags_saturatedCenter",
        "diaObjectForcedSource.invalidPsfFlag",
    ],
    filters=[("nDiaSources", ">=", 20)],
)
dia_object = dia_object.query(
    "diaObjectForcedSource.psfDiffFlux.notna()"
    " & diaObjectForcedSource.psfDiffFluxErr_corrected.notna()"
    " & ~diaObjectForcedSource.psfDiffFlux_flag"
    " & ~diaObjectForcedSource.psfDiffFluxErr_corrected_flag"
    " & ~diaObjectForcedSource.diff_PixelFlags_nodataCenter"
    " & ~diaObjectForcedSource.pixelFlags_saturatedCenter"
    " & ~diaObjectForcedSource.invalidPsfFlag"
)
dia_object = dia_object.query("diaObjectForcedSource.len() >= 20")


def prepare_light_curves(df):
    """Sort observations by time and replace MJD with a float32 time column."""
    df = df.sort_values("diaObjectForcedSource.midpointMjdTai")
    df["diaObjectForcedSource.time"] = (df["diaObjectForcedSource.midpointMjdTai"] - 60_000).astype(
        np.float32
    )
    return df.drop(columns=["diaObjectForcedSource.midpointMjdTai"])


dia_object = dia_object.map_partitions(prepare_light_curves)

We are looking for objects like a transient, so we keep only strongly variable objects with large variability amplitude. The χ² p-value of the light curve being constant must be below 10⁻⁶ in each of g, r, i, z bands, and the interpercentile range between the 5th and 95th percentiles of the flux must be larger than 1 μJy in at least one of these bands.

In [ ]:
BANDS = ["u", "g", "r", "i", "z", "y"]
variability_features = licu.Extractor(
    licu.Chi2Pvar(bands=BANDS),
    licu.InterPercentileRange(0.05, bands=BANDS),
)


def add_variability_features(df):
    """Add chi2 p-value and interpercentile range columns for each band."""
    # Extractor.many() doesn't accept an empty list of light curves
    if len(df) == 0:
        return df.assign(**{name: np.array([], dtype=np.float64) for name in variability_features.names})
    values = variability_features.many(
        pa.array(df["diaObjectForcedSource"]),
        arrow_fields={"t": "time", "m": "psfDiffFlux", "sigma": "psfDiffFluxErr_corrected", "band": "band"},
        fill_value=np.nan,
        # 4 workers x 4 threads = 16 CPUs
        n_jobs=4,
    )
    return df.assign(**dict(zip(variability_features.names, values.T)))


dia_object = dia_object.map_partitions(add_variability_features)
chi2_query = " and ".join(f"chi2_pvar_{band} < 1e-6" for band in "griz")
# Fluxes are in nJy
amplitude_query = " or ".join(f"inter_percentile_range_5_{band} > 1000" for band in "griz")
variability_query = f"{chi2_query} and ({amplitude_query})"
print(variability_query)

dia_object = dia_object.query(variability_query)

### Light-curve embeddings

We use ATCAT, a transformer model trained on LSST-like multiband light curves, from the `light-curve` package ([docs](https://light-curve.snad.space/latest/embed/)). ATCAT processes up to 243 observations at once, so longer light curves are split into non-overlapping windows, and the embeddings of the windows are averaged.

We embed light curves with `map_rows()` inside `map_partitions()`. ONNX sessions can't be pickled and sent to the Dask workers, so we create the model in each partition: it is downloaded once and then loaded from the local cache.

In [ ]:
BAND_GROUPS = {band: i for i, band in enumerate(BANDS)}


def embed_partition(df):
    """ATCAT embeddings of the light curves of a single partition, as a nested column."""
    # Skip model loading for empty partitions, e.g. when Dask infers the output schema
    if len(df) == 0:
        return df[["diaObjectId"]].assign(
            embedding=pd.Series([], dtype=npd.NestedDtype({"value": pa.float32()}))
        )
    model = ATCAT.from_hf(output="mean", band_groups=BAND_GROUPS, reduction="non-overlapping-windows")

    def embed(time, flux, flux_err, band):
        return {"embedding.value": model(time, flux, flux_err, band).squeeze()}

    embeddings = df.map_rows(
        embed,
        columns=[
            "diaObjectForcedSource.time",
            "diaObjectForcedSource.psfDiffFlux",
            "diaObjectForcedSource.psfDiffFluxErr_corrected",
            "diaObjectForcedSource.band",
        ],
        row_container="args",
    )
    return df[["diaObjectId"]].join(embeddings)

### Find the query object

We look for the DiaObject within 1 arcsec of the TNS coordinates, and print its ID only.

In [ ]:
query = (
    dia_object.search(lsdb.ConeSearch(ra=QUERY_RA, dec=QUERY_DEC, radius_arcsec=1.0))
    .map_partitions(embed_partition)
    .compute()
)
assert len(query) == 1, f"Expected exactly one DiaObject, found {len(query)}"
query_id = query["diaObjectId"].iloc[0]
print(f"{QUERY_NAME}: diaObjectId {query_id}")

### Embed light curves around the query object

To keep the computation short, we search in a 0.6-degree cone around the query object.

In [ ]:
search_cone = lsdb.ConeSearch(ra=QUERY_RA, dec=QUERY_DEC, radius_arcsec=0.6 * 3600)

embeddings = dia_object.search(search_cone).map_partitions(embed_partition).compute()
embeddings = embeddings[embeddings["diaObjectId"] != query_id].reset_index(drop=True)
print(f"Number of embedded objects: {len(embeddings)}")

### Find the nearest neighbours

We use the cosine distance between the embeddings: the smaller the distance, the more similar the light curves.

In [ ]:
N_NEIGHBORS = 10

# Stack the nested embeddings into (N, 384) matrices
query_matrix = np.asarray(query["embedding.value"]).reshape(len(query), -1)
embedding_matrix = np.asarray(embeddings["embedding.value"]).reshape(len(embeddings), -1)

embeddings["cosine_distance"] = cdist(query_matrix, embedding_matrix, metric="cosine")[0]
neighbors = embeddings.nsmallest(N_NEIGHBORS, "cosine_distance")
neighbors[["diaObjectId", "cosine_distance"]]

Is the nearest neighbour a known object? There is another SN Ia candidate from the [TNS EDP2 explorer](https://trivialtz.github.io/tns-edp2-explorer/) in this region, [AT2025agtf](https://www.wis-tns.org/object/2025agtf). We look for its DiaObject in the same way as for the query object, and check its rank among the neighbours.

In [ ]:
agtf = dia_object.search(lsdb.ConeSearch(ra=9.35233, dec=-42.46275, radius_arcsec=1.0)).compute()
agtf_id = agtf["diaObjectId"].iloc[0]
rank = embeddings["cosine_distance"].rank()[embeddings["diaObjectId"] == agtf_id].iloc[0]
print(f"AT2025agtf: diaObjectId {agtf_id}, rank {rank:.0f} of {len(embeddings)}")

### Share back the results

We save the DiaObject IDs, cosine distances and embeddings of the nearest neighbours as a plain Parquet file: no coordinates, no `_healpix_29`, and no light curves.

In [ ]:
neighbors[["diaObjectId", "cosine_distance", "embedding"]].to_parquet(f"similar_to_{QUERY_NAME}.parquet")
npd.read_parquet(f"similar_to_{QUERY_NAME}.parquet").head()

In [ ]:
client.close()